# Model files: compiling a `.model` file to Fortran

The `thermodiff.model` front-end lets you describe a thermodynamic model in a
small text file and compile it into a Fortran module with the function and all
its analytical derivatives. This is the recommended entry point if you do not
want to write Python. The `.model` file is the single source of truth.

In [1]:
# Install if you are in Google Colab
import sys

if "google.colab" in sys.modules:
    !pip install git+https://github.com/SalvadorBrandolin/thermodiff

## 1. Describe the model

A `.model` file is organized in sections: `variables`, `parameters`,
`expressions` and `function`. Intermediate expressions are written in index
notation and substituted into the final `function` before it is differentiated.
Here is a Peng-Robinson style pressure model:

In [2]:
model = r'''
model pr
target fortran
use yaeos__constants, only: R

variables
    n   nc
    V
    T

parameters
    ac    nc
    Tc    nc
    k     nc nc
    b     nc

expressions
    alpha_i = (T/Tc_i)^2
    a_i     = ac_i*alpha_i
    amix    = sum_i sum_j n_i*n_j*sqrt(a_i*a_j)*(1 - k_ij)
    bmix    = sum_i n_i*b_i

function
    R*T/(V - bmix) - amix/(V*(V + bmix) + bmix*(V - bmix))
'''
print(model)


model pr
target fortran
use yaeos__constants, only: R

variables
    n   nc
    V
    T

parameters
    ac    nc
    Tc    nc
    k     nc nc
    b     nc

expressions
    alpha_i = (T/Tc_i)^2
    a_i     = ac_i*alpha_i
    amix    = sum_i sum_j n_i*n_j*sqrt(a_i*a_j)*(1 - k_ij)
    bmix    = sum_i n_i*b_i

function
    R*T/(V - bmix) - amix/(V*(V + bmix) + bmix*(V - bmix))



## 2. Compile it

`compile_model` returns the Fortran source of a module that contains a
subroutine evaluating the function and all first and second order derivatives.
On the command line the same is done with
`thermodiff-model pr.model -o pr.f90`.

In [3]:
from thermodiff.model import compile_model

fortran = compile_model(model, filename="pr.model")
print(fortran)

! Generated by thermodiff from model 'pr'.
! Do not edit by hand.
module pr_mod
   use yaeos__constants, only: R
   implicit none
contains

   subroutine pr_derivs(nc, V, T, Tc, ac, b, k, n, f, dfdT, dfdV, dfdn, dfdT2, dfdV2, dfdndn, dfdTdV, dfdndT, dfdndV)
      implicit none
      integer, intent(in) :: nc
      real(8), intent(in) :: V
      real(8), intent(in) :: T
      real(8), intent(in) :: Tc(:)
      real(8), intent(in) :: ac(:)
      real(8), intent(in) :: b(:)
      real(8), intent(in) :: k(:, :)
      real(8), intent(in) :: n(:)
      real(8), intent(out) :: f
      real(8), intent(out) :: dfdT
      real(8), intent(out) :: dfdV
      real(8), intent(out) :: dfdn(:)
      real(8), intent(out) :: dfdT2
      real(8), intent(out) :: dfdV2
      real(8), intent(out) :: dfdndn(:, :)
      real(8), intent(out) :: dfdTdV
      real(8), intent(out) :: dfdndT(:)
      real(8), intent(out) :: dfdndV(:)
      real(8) :: res, t1, t10, t11, t12, t13, t14, t15, t16, t17, t18, t19, t2, t

A few things to notice:

* The `use yaeos__constants, only: R` line is preserved, so `R` is taken from
  the `yaeos` module instead of being passed as an argument.
* `nc` and the rank of every array are inferred from the declarations;
  `k` is declared as `k(:, :)` and `b` as `b(:)`.
* The intermediate expressions `alpha`, `a`, `amix` and `bmix` were inlined
  and do not appear in the generated code.
* All the first and second order derivatives of the declared variables are
  emitted: `f`, `dfdT`, `dfdV`, `dfdT2`, `dfdV2`, `dfdTdV`, `dfdn`, `dfdndn`,
  `dfdndT` and `dfdndV`.

## 3. Verify the derivatives numerically

The generated subroutine can be compiled and compared with central finite
differences of the model. The next cell does that when `gfortran` is
available.

In [4]:
import os
import shutil
import subprocess
import tempfile

compiler = shutil.which("gfortran")

stub = (
    "module yaeos__constants\n"
    "  implicit none\n"
    "  real(8), parameter :: R = 8.31446261815324d0\n"
    "end module yaeos__constants\n"
)

driver = '''
program main
  use pr_mod
  implicit none
  integer, parameter :: nc = 2
  real(8) :: n(nc), V, T, ac(nc), Tc(nc), k(nc, nc), b(nc)
  real(8) :: f, dfdT, dfdV, dfdT2, dfdV2, dfdTdV
  real(8) :: dfdn(nc), dfdndn(nc, nc), dfdndT(nc), dfdndV(nc)
  integer :: i
  n = [1.0d0, 2.0d0]
  ac = [0.5d0, 0.8d0]
  Tc = [300.0d0, 400.0d0]
  b = [0.04d0, 0.03d0]
  k = reshape([0.0d0, 0.02d0, 0.02d0, 0.0d0], [nc, nc])
  V = 3.0d0
  T = 320.0d0
  call pr_derivs(nc, V, T, Tc, ac, b, k, n, f, dfdT, dfdV, dfdn, &
                 dfdT2, dfdV2, dfdndn, dfdTdV, dfdndT, dfdndV)
  write(*,*) 'f      =', f
  write(*,*) 'dfdT   =', dfdT
  write(*,*) 'dfdV   =', dfdV
  write(*,*) 'dfdn   =', (dfdn(i), i=1,nc)
  write(*,*) 'dfdndn =', (dfdndn(i,i), i=1,nc)
end program main
'''

if compiler is None:
    print("gfortran not found, skipping the numerical check")
else:
    workdir = tempfile.mkdtemp()
    source = os.path.join(workdir, "main.f90")
    with open(source, "w") as handle:
        handle.write(stub + fortran + driver)
    exe = os.path.join(workdir, "a.out")
    subprocess.run(
        [compiler, "-O2", "-o", exe, source], check=True, cwd=workdir
    )
    print(subprocess.run([exe], capture_output=True, text=True).stdout)

 f      =   916.96446181898011     
 dfdT   =   2.8639718111433705     
 dfdV   =  -316.04576867931746     
 dfdn   =   12.327283070984604        9.1760394474430420     
 dfdndn =  0.24612187652112627       0.10092104812048516     



## Notes

* The module is named `<model>_mod` and the subroutine `<model>_derivs`.
* Indexed names use a single letter per index (`n_i`, `k_ij`); the
  compiler renames them internally so they never clash with the
  differentiation indices.
* Sums are written `sum_i expr` (from `1` to `nc`) or with explicit
  limits `sum_i[a:b] expr`.
* Every undeclared name, mismatched subscript or unbalanced index is
  reported with the file and line number.